# 05 — Evaluation

Compare the baseline (sklearn) and CNN (PyTorch) models side-by-side.

1. Run `evaluate_model` for both baseline and CNN.
2. `compare_models()` — display comparison table.
3. Precision-Recall and ROC curves plotted together.

> Falls back to synthetic predictions when trained models are unavailable.

In [ ]:
import os, sys, tempfile, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    precision_recall_curve, average_precision_score,
    roc_curve, roc_auc_score,
)

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import src.evaluate as ev

MANIFEST = os.path.join(PROJECT_ROOT, "data", "manifest.csv")
FEATURE_DIR = os.path.join(PROJECT_ROOT, "data", "features")
SPEC_DIR = os.path.join(PROJECT_ROOT, "data", "spectrograms")
BASELINE_MODEL = os.path.join(PROJECT_ROOT, "models", "baseline", "model.pkl")
CNN_MODEL = os.path.join(PROJECT_ROOT, "models", "cnn", "best.pt")

print(f"Project root: {PROJECT_ROOT}")

## Detect available models (or use synthetic fallback)

In [ ]:
HAVE_BASELINE = os.path.isfile(BASELINE_MODEL) and os.path.isfile(MANIFEST)
HAVE_CNN = os.path.isfile(CNN_MODEL) and os.path.isfile(MANIFEST)
USE_SYNTHETIC = not (HAVE_BASELINE and HAVE_CNN)

print(f"Baseline model available: {HAVE_BASELINE}")
print(f"CNN model available     : {HAVE_CNN}")

if USE_SYNTHETIC:
    print("\n⚠️  Using synthetic predictions for demonstration.")
    np.random.seed(42)
    _y_true = np.array([1, 0, 1, 0, 1, 0, 1, 0, 1, 0])

    def _fake_baseline(*a, **kw):
        y_pred = np.array([1, 0, 1, 1, 1, 0, 0, 0, 1, 0])
        y_prob = np.array([0.91, 0.12, 0.87, 0.62, 0.95, 0.18, 0.35, 0.10, 0.88, 0.22])
        return _y_true.copy(), y_pred, y_prob

    def _fake_cnn(*a, **kw):
        y_pred = np.array([1, 0, 1, 0, 1, 0, 1, 1, 1, 0])
        y_prob = np.array([0.96, 0.04, 0.93, 0.15, 0.97, 0.08, 0.80, 0.55, 0.92, 0.11])
        return _y_true.copy(), y_pred, y_prob

    ev._infer_baseline = _fake_baseline
    ev._infer_cnn = _fake_cnn

## 1. Evaluate both models

In [ ]:
TMP_OUT = tempfile.mkdtemp(prefix="eval_nb_")

try:
    baseline_metrics = ev.evaluate_model(
        model_type="baseline",
        model_path=BASELINE_MODEL if HAVE_BASELINE else "dummy.pkl",
        manifest_csv=MANIFEST if HAVE_BASELINE else "dummy.csv",
        feature_dir=FEATURE_DIR,
        output_dir=TMP_OUT,
    )
except Exception as e:
    print(f"⚠️  Baseline evaluation failed: {e}")
    baseline_metrics = None

In [ ]:
try:
    cnn_metrics = ev.evaluate_model(
        model_type="cnn",
        model_path=CNN_MODEL if HAVE_CNN else "dummy.pt",
        manifest_csv=MANIFEST if HAVE_CNN else "dummy.csv",
        spec_dir=SPEC_DIR,
        output_dir=TMP_OUT,
    )
except Exception as e:
    print(f"⚠️  CNN evaluation failed: {e}")
    cnn_metrics = None

## 2. Side-by-side comparison table

In [ ]:
if baseline_metrics and cnn_metrics:
    comparison_df = ev.compare_models(baseline_metrics, cnn_metrics)
    display(comparison_df)
else:
    print("⚠️  Cannot compare — one or both evaluations failed.")

## 3. PR & ROC curves together on one figure

In [ ]:
try:
    if USE_SYNTHETIC:
        y_true_b, _, y_prob_b = _fake_baseline()
        y_true_c, _, y_prob_c = _fake_cnn()
    else:
        y_true_b, _, y_prob_b = ev._infer_baseline(
            BASELINE_MODEL, MANIFEST, FEATURE_DIR)
        y_true_c, _, y_prob_c = ev._infer_cnn(
            CNN_MODEL, MANIFEST, SPEC_DIR)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Precision-Recall
    for y_true, y_prob, name, color in [
        (y_true_b, y_prob_b, "Baseline", "#4c72b0"),
        (y_true_c, y_prob_c, "CNN", "#c44e52"),
    ]:
        prec, rec, _ = precision_recall_curve(y_true, y_prob)
        ap = average_precision_score(y_true, y_prob)
        ax1.plot(rec, prec, lw=2, color=color,
                 label=f"{name} (AP={ap:.3f})")
    ax1.set_xlabel("Recall")
    ax1.set_ylabel("Precision")
    ax1.set_title("Precision-Recall Curves")
    ax1.legend(loc="lower left")
    ax1.grid(True, alpha=0.3)

    # ROC
    for y_true, y_prob, name, color in [
        (y_true_b, y_prob_b, "Baseline", "#4c72b0"),
        (y_true_c, y_prob_c, "CNN", "#c44e52"),
    ]:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc = roc_auc_score(y_true, y_prob)
        ax2.plot(fpr, tpr, lw=2, color=color,
                 label=f"{name} (AUC={auc:.3f})")
    ax2.plot([0, 1], [0, 1], "k--", alpha=0.3)
    ax2.set_xlabel("False Positive Rate")
    ax2.set_ylabel("True Positive Rate")
    ax2.set_title("ROC Curves")
    ax2.legend(loc="lower right")
    ax2.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()
except Exception as e:
    print(f"⚠️  Could not plot curves: {e}")

In [ ]:
# Clean up
if os.path.isdir(TMP_OUT):
    shutil.rmtree(TMP_OUT)
    print(f"Cleaned up {TMP_OUT}")